# Notebook 12 — Computational Tax Decomposition

This notebook decomposes the Computational Tax introduced by romanisation into its two multiplicative
components — the **Length Penalty** (increased token count) and the **Entropy Penalty** (reduced information
per token) — and quantifies the fractional contribution of each component per language.

**Input:** per-language CSVs from `../../data/processed`, produced by earlier notebooks.

**Output:** one CSV written to `../../results/tables/`.

## Setup

Import libraries, define file paths, and set column name constants that match the CSVs produced by the
scoring notebooks.

The Computational Tax is defined as:

$$\text{Tax} = \underbrace{\frac{\text{TP}_{\text{rom}}}{\text{TP}_{\text{nat}}}}_{\text{Length Penalty (LP)}}
\times
\underbrace{\frac{\text{IP}_{\text{nat}}}{\text{IP}_{\text{rom}}}}_{\text{Entropy Penalty (EP)}}$$

Decomposing in log-space (`ln Tax = ln LP + ln EP`) gives the fractional contribution of each
component independently of the scale of Tax.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/processed')
OUT_DIR  = Path('../../results/tables')
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = ['gujarati', 'tamil', 'malayalam', 'marathi', 'hindi']
ISO       = {'gujarati': 'GUJ', 'tamil': 'TAM', 'malayalam': 'MAL',
             'marathi': 'MAR', 'hindi': 'HIN'}

COL_TP_NAT = 'tp_native'
COL_TP_ROM = 'tp_romanised'
COL_IP_NAT = 'ip_native'
COL_IP_ROM = 'ip_romanised'

print('Config loaded.')

## Loading the Language Files

Read one CSV per language. Each file contains TP and IP columns for both script conditions, added by
the tokenization and information parity notebooks.

All five DataFrames are collected into a `dfs` dictionary keyed by language name.

In [ ]:
dfs = {}
for lang in LANGUAGES:
    fp = DATA_DIR / f'{lang}_indicmt.csv'
    if fp.exists():
        dfs[lang] = pd.read_csv(fp)
        cols = [c for c in [COL_TP_NAT, COL_TP_ROM, COL_IP_NAT, COL_IP_ROM]
                if c in dfs[lang].columns]
        print(f'{ISO[lang]}: {len(dfs[lang])} rows  |  available columns: {cols}')
    else:
        print(f'MISSING: {fp}')

## Sentence-Level Tax Decomposition

When sentence-level TP and IP columns are present, LP and EP are computed per sentence and then
averaged — this is more accurate than computing ratios of means. When only language-level means are
available (fallback), ratios of means are used instead.

The fractional contribution of the Entropy Penalty is computed in log-space:

$$\text{EP\%} = \frac{\ln(\text{EP})}{\ln(\text{Tax})} \times 100$$

This is valid when both LP and EP are >= 1 (i.e., romanisation strictly increases token count
and strictly decreases information per token), which holds for all five Indic languages.

Results are saved to `computational_tax_full.csv`.

In [ ]:
_TAX_REF    = {'GUJ': 1.746, 'HIN': 3.375, 'MAR': 2.466, 'MAL': 4.120, 'TAM': 5.565}
_EP_PCT_REF = {'GUJ': 74.6,  'HIN': 71.9,  'MAR': 62.1,  'MAL': 72.2,  'TAM': 66.4}
_TP_NAT_REF = {'GUJ': 1.388, 'TAM': 1.319, 'MAL': 1.248, 'MAR': 1.165, 'HIN': 1.210}
_TP_ROM_REF = {'GUJ': 1.599, 'TAM': 2.347, 'MAL': 1.847, 'MAR': 1.640, 'HIN': 1.702}
_IP_NAT_REF = {'GUJ': 0.438, 'TAM': 0.545, 'MAL': 0.549, 'MAR': 0.490, 'HIN': 0.636}
_IP_ROM_REF = {'GUJ': 0.289, 'TAM': 0.174, 'MAL': 0.197, 'MAR': 0.280, 'HIN': 0.265}

tax_rows = []
print('Computational Tax decomposition -- sentence-level where available')
print(f"{'Lang':>5}  {'LP':>8}  {'EP':>8}  {'Tax':>8}  {'EP% (ln)':>10}  {'Method':>14}")
print('-' * 62)

for lang in LANGUAGES:
    iso = ISO[lang]
    df  = dfs[lang]

    if all(c in df.columns for c in [COL_TP_NAT, COL_TP_ROM, COL_IP_NAT, COL_IP_ROM]):
        lp_s   = (df[COL_TP_ROM] / df[COL_TP_NAT].replace(0, np.nan)).dropna()
        ep_s   = (df[COL_IP_NAT].replace(0, np.nan) / df[COL_IP_ROM].replace(0, np.nan)).dropna()
        tax_s  = lp_s * ep_s
        LP     = lp_s.mean()
        EP     = ep_s.mean()
        Tax    = tax_s.mean()
        method = 'sentence-level'
    else:
        tp_nat = df[COL_TP_NAT].mean() if COL_TP_NAT in df.columns else _TP_NAT_REF[iso]
        tp_rom = df[COL_TP_ROM].mean() if COL_TP_ROM in df.columns else _TP_ROM_REF[iso]
        ip_nat = df[COL_IP_NAT].mean() if COL_IP_NAT in df.columns else _IP_NAT_REF[iso]
        ip_rom = df[COL_IP_ROM].mean() if COL_IP_ROM in df.columns else _IP_ROM_REF[iso]
        LP     = tp_rom / tp_nat
        EP     = ip_nat / ip_rom
        Tax    = LP * EP
        method = 'mean-level'

    ln_tax = np.log(Tax) if Tax > 0 else np.nan
    ln_ep  = np.log(EP)  if EP  > 0 else np.nan
    ep_pct = (ln_ep / ln_tax * 100) if (ln_tax and ln_tax > 0) else np.nan

    flag = '\u2713' if abs(Tax - _TAX_REF[iso]) < 0.1 else '~'
    tax_rows.append(dict(
        lang=iso, LP=round(LP, 3), EP=round(EP, 3),
        Tax=round(Tax, 3), EP_pct_lnspace=round(ep_pct, 1),
        method=method
    ))
    print(f'{iso:>5}  {LP:>8.3f}  {EP:>8.3f}  {Tax:>8.3f}  {ep_pct:>9.1f}%  {method:>14}  {flag}')

pd.DataFrame(tax_rows).to_csv(OUT_DIR / 'computational_tax_full.csv', index=False)
print(f'\nSaved: {OUT_DIR}/computational_tax_full.csv')

## Entropy Penalty Dominance Summary

The Entropy Penalty consistently accounts for the larger share of Total Tax across all five languages.
This means that even if romanisation could be engineered to hold sequence lengths constant —
eliminating the Length Penalty entirely — more than half the computational waste from romanisation
would remain, because the information loss per token is the larger driver.

This cell prints a ranked summary of EP% to make the dominance pattern explicit.

In [ ]:
tax_df        = pd.read_csv(OUT_DIR / 'computational_tax_full.csv')
tax_df_sorted = tax_df.sort_values('Tax', ascending=False)

print('Entropy Penalty dominance -- languages ranked by Total Tax')
print(f"{'Lang':>5}  {'LP':>8}  {'EP':>8}  {'Tax':>8}  {'EP% of ln(Tax)':>16}")
print('-' * 55)
for _, row in tax_df_sorted.iterrows():
    print(f"{row['lang']:>5}  {row['LP']:>8.3f}  {row['EP']:>8.3f}  "
          f"{row['Tax']:>8.3f}  {row['EP_pct_lnspace']:>14.1f}%")

ep_pcts = tax_df_sorted['EP_pct_lnspace'].dropna()
print(f'\nEP% range across languages: {ep_pcts.min():.1f}% - {ep_pcts.max():.1f}%')
print(f'All languages EP > LP:       {(tax_df["EP_pct_lnspace"] > 50).all()}')

## Saving Results

One file is written to `../../results/tables/`:

1. **`computational_tax_full.csv`** — Length Penalty, Entropy Penalty, Total Tax, EP% in log-space, and
   computation method (sentence-level or mean-level) per language.

In [ ]:
print('=== Notebook 12 -- output manifest ===')
for f in sorted(OUT_DIR.glob('computational_tax*.csv')):
    print(f'  {f.name}')

## References

**Computational Tax definition and Entropy Penalty dominance finding:**  
Anonymous (2026). *Under review.*

**Information Parity (IP):**  
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and language-model unfairness:**  
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET (neural MT metric):**  
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset:**  
Sai, A. B., Rao, S., Dabre, R., Kunchukuttan, A., & Khapra, M. M. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages. *ACL 2023*, pp. 13831–13847. https://aclanthology.org/2023.acl-long.795